---
# How This Code Works
---
This notebook is a guided tour of the tutorial codebase.

It is written for a reader who wants to understand the code slowly, calmly, and without needing to guess how the files fit together. The goal is not to impress you with technical language. The goal is to help you feel oriented.

If you ever feel lost while reading the code, come back to this notebook and use it as a map.

**Project and authors**
- Co-authors: [Andre Cire](https://utsc.utoronto.ca/mgmt/andre-cire), [Selva Nadarajah](https://selvan.people.uic.edu/), [Parshan Pakiman](https://parshanpakiman.github.io/), [Negar Soheili](https://www.negar-soheili.com/)
- GitHub: [self-adapting-mdp-approximations/informs-tutorials](https://github.com/self-adapting-mdp-approximations/informs-tutorials)



---
## Table of Contents

1. [One-Minute Big Picture](#big-picture)
2. [What Each File Is For](#file-map)
3. [The Usual Order To Read The Code](#reading-order)
4. [Configuration File: `config.py`](#tutorial-config)
5. [MDP File: `mdp.py`](#mdp-file)
6. [Basis File: `basis.py`](#basis-file)
7. [FALP File: `Self_Guided_ALP/falp.py`](#falp-file)
8. [SGALP File: `Self_Guided_ALP/sgalp.py`](#sgalp-file)
9. [Lower-Bound File: `Self_Guided_ALP/cvl_lower_bound.py`](#lower-bound-file)
10. [Policy File: `policy.py`](#policy-file)
11. [Helper File: `helper.py`](#helper-file)
12. [Experiment Notebook: `Notebooks/tutorial_self_guided_ALP.ipynb`](#tutorial-notebook)
13. [Short End-to-End Story](#end-to-end-story)


---
<a id="big-picture"></a>
## 1. One-Minute Big Picture

This project answers one basic question:

**How can we approximate the long-run cost of an inventory system when solving the exact dynamic program is hard?**

The code does that in seven steps:

1. `mdp.py` defines the inventory problem itself.
2. `basis.py` defines a flexible family of curves that can approximate the value function.
3. `Self_Guided_ALP/falp.py` fits one approximate value function using sampled linear constraints.
4. `Self_Guided_ALP/sgalp.py` improves on that idea by growing the basis gradually and guiding each new stage.
5. `Self_Guided_ALP/cvl_lower_bound.py` estimates a lower bound, which is a performance certificate.
6. `policy.py` estimates an upper bound by simulating a greedy policy.
7. `helper.py` ties everything together for experiments and plots.

So the codebase is not random. It is a pipeline.


<a id="file-map"></a>
## 2. What Each File Is For

| Path | Main job | Simple description |
| --- | --- | --- |
| `config.py` | stores grouped parameters | “Where do all the knobs live?” |
| `mdp.py` | defines the inventory problem | “What system are we controlling?” |
| `basis.py` | builds basis functions | “What shapes can the approximate value function take?” |
| `Self_Guided_ALP/falp.py` | fits one FALP model | “How do we solve one approximate LP?” |
| `Self_Guided_ALP/sgalp.py` | fits staged self-guided ALP | “How do we improve the approximation gradually?” |
| `Self_Guided_ALP/cvl_lower_bound.py` | estimates a lower bound | “How good can we certify the approximation is?” |
| `PSMD/psmd.py` | fits the tutorial PSMD baseline | “How do we learn from violated Bellman constraints iteratively?” |
| `policy.py` | estimates an upper bound | “How costly is the greedy policy in simulation?” |
| `helper.py` | runs experiments and plots | “How do we automate repeated runs?” |
| `Notebooks/tutorial_self_guided_ALP.ipynb` | FALP and SGALP tutorial notebook | “How do we show ALP results clearly?” |
| `Notebooks/tutorial_psmd.ipynb` | PSMD tutorial notebook | “How do we show PSMD results clearly?” |
| `Notebooks/how_code_work.ipynb` | reading guide | “How do I understand the project?” |


---
<a id="reading-order"></a>
## 3. The Usual Order To Read The Code

If you try to read the files in the wrong order, the project can feel harder than it really is.

A beginner-friendly order is:

1. Start with `config.py`.
   This tells you what kinds of choices the code exposes.
2. Then read `mdp.py`.
   This tells you what the actual inventory model is.
3. Then read `basis.py`.
   This tells you how the value function will be approximated.
4. Then read `Self_Guided_ALP/falp.py`.
   This is the simpler algorithm.
5. Then read `Self_Guided_ALP/sgalp.py`.
   This is the more advanced algorithm built on the same ingredients.
6. Then read `policy.py` and `Self_Guided_ALP/cvl_lower_bound.py`.
   These files evaluate the quality of the fitted models.
7. Read `helper.py` last.
   It is not the best place to learn first principles. It is the best place to see how the pieces are orchestrated.

Now that the project has folders, a helpful mental model is: `Self_Guided_ALP/` holds the ALP-specific logic, `Notebooks/` holds the explanatory notebooks, and the main folder holds reusable shared modules.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start_path: Path) -> Path:
    """
    Find the tutorial project root by looking for the shared Python modules.

    Args:
        start_path: Directory from which to begin the upward search.
    """
    for candidate in (start_path, *start_path.parents):
        if (candidate / "helper.py").exists() and (candidate / "config.py").exists():
            return candidate
    raise RuntimeError("Could not locate the tutorial project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("Main folder:")
for path in sorted(PROJECT_ROOT.glob("*.py")):
    print("  ", path.name)

print("\nSelf_Guided_ALP package:")
for path in sorted((PROJECT_ROOT / "Self_Guided_ALP").glob("*.py")):
    print("  ", f"Self_Guided_ALP/{path.name}")

print("\nPSMD package:")
for path in sorted((PROJECT_ROOT / "PSMD").glob("*.py")):
    print("  ", f"PSMD/{path.name}")

print("\nNotebook folder:")
for path in sorted((PROJECT_ROOT / "Notebooks").glob("*.ipynb")):
    print("  ", f"Notebooks/{path.name}")


---
<a id="tutorial-config"></a>
## 4. Configuration File: `config.py`

This file exists for one very practical reason:

**without it, constructors and helper functions would have too many separate arguments.**

So instead of passing many loose numbers around, the code groups related choices into small config classes.

This is the file that answers questions like:
- How many random features are we using?
- How many constraints do we sample?
- What seed do we use?
- How many trajectories do we simulate?
- What are the inventory-cost parameters?


### `ConfigMixin`

Purpose: adds the `with_updates(...)` convenience method.

Why this helps:
- you can make a base config once,
- then create a slightly changed version without rebuilding everything from scratch.

Example idea:
- “Use the same FALP settings, but change the number of features from 2 to 3.”


### `RandomFeatureConfig`

Purpose: controls how the random Fourier basis is generated.

| Parameter | Meaning in plain language |
| --- | --- |
| `bandwidth_choices` | candidate scales for how wiggly or smooth the basis functions are |
| `random_seed` | the seed used to sample the random basis functions |

Why it matters:
- this config changes the family of curves available to approximate the value function.


### `HiGHSSolverConfig`

Purpose: stores numerical settings for the LP solver.

| Parameter | Meaning in plain language |
| --- | --- |
| `method` | which HiGHS solver variant SciPy should use |
| `primal_feasibility_tolerance` | how strict the solver is when checking whether constraints are satisfied |
| `dual_feasibility_tolerance` | how strict the solver is in the dual calculations |

If you are only trying to understand the project conceptually, you can treat this as a “numerical housekeeping” config.


### `GuidingConstraintConfig`

Purpose: stores the extra settings used only by self-guided ALP.

| Parameter | Meaning in plain language |
| --- | --- |
| `num_guiding_states` | how many states are used to build guiding constraints |
| `allowed_violation` | how much the new stage is allowed to fall below the old approximation |
| `relax_fraction` | a relative cushion used to make guiding constraints less brittle |
| `absolute_floor` | a minimum cushion so the allowance never becomes too tiny |
| `retry_scales` | backup relaxation levels tried if the LP is numerically difficult |

This config is important because SGALP’s extra behavior lives here.


### `FALPConfig`

Purpose: collects the main settings for one FALP model.

| Parameter | Meaning in plain language |
| --- | --- |
| `num_random_features` | how many random basis functions to use |
| `num_constraints` | how many sampled Bellman constraints to build |
| `num_state_relevance_samples` | how many sampled states to use in the FALP objective |
| `random_features` | the `RandomFeatureConfig` object used to build the basis |
| `solver` | whether to use SciPy or the tiny teaching fallback solver |

If someone asks, “What defines one FALP experiment?”, this config is the shortest answer.


### `SGALPConfig`

Purpose: collects the main settings for one SGALP model.

| Parameter | Meaning in plain language |
| --- | --- |
| `max_random_features` | the largest number of random features SGALP is allowed to reach |
| `batch_size` | how many features to add at each stage |
| `num_constraints` | how many sampled Bellman constraints to build at each stage |
| `num_state_relevance_samples` | how many sampled states to use in the objective |
| `random_features` | the `RandomFeatureConfig` used for the shared basis family |
| `guiding` | the `GuidingConstraintConfig` for SGALP’s extra constraints |
| `solver` | the `HiGHSSolverConfig` for LP solution details |

This is basically “FALP config plus SGALP-specific guidance settings.”


### `LowerBoundConfig`

Purpose: stores all settings for the lower-bound estimator.

| Parameter | Meaning in plain language |
| --- | --- |
| `num_mc_init_states` | how many starting points the sampler begins from |
| `chain_length` | how many sampling steps to run |
| `burn_in` | how many early samples to discard |
| `proposal_state_std` | step size when proposing new states |
| `proposal_action_std` | step size when proposing new actions |
| `random_seed` | randomness seed for the lower-bound estimator |
| `noise_batch_size` | how many demand samples to reuse inside the Bellman residual calculation |
| `sampler` | whether to use `emcee` or the built-in fallback sampler |
| `num_walkers` | number of walkers when using `emcee` |
| `initial_state` | the initial inventory level at which the lower bound is interpreted |

This config matters after a model has already been fitted.


### `PolicyEvaluationConfig`

Purpose: stores settings for the approximate greedy policy and Monte Carlo simulation.

| Parameter | Meaning in plain language |
| --- | --- |
| `state_grid_size` | how finely to discretize the state axis when precomputing the greedy policy |
| `policy_noise_batch_size` | how many demand samples to use when evaluating each candidate action |
| `policy_noise_seed` | seed for the policy lookup calculation |
| `num_trajectories` | how many simulated trajectories to average over |
| `horizon` | how long each simulation runs |
| `simulation_seed` | seed for the simulated demand paths |
| `initial_state` | starting inventory level in policy simulation |

This config produces the project’s upper-bound estimate.


### `InventoryMDPConfig`

Purpose: stores the problem data for the inventory model.

| Parameter | Meaning in plain language |
| --- | --- |
| `mdp_name` | simple label for the model |
| `discount` | how much future cost matters relative to current cost |
| `random_seed` | base seed used for the MDP’s sampling routines |
| `lower_state_bound` | smallest allowed inventory state |
| `upper_state_bound` | largest allowed inventory state |
| `max_order` | largest order quantity allowed |
| `purchase_cost` | cost per ordered unit |
| `holding_cost` | cost for ending with positive inventory |
| `backlog_cost` | cost for ending with negative inventory |
| `disposal_cost` | penalty for inventory above the upper bound |
| `lost_sale_cost` | penalty for inventory below the lower bound |
| `demand_mean` | average demand |
| `demand_std` | demand standard deviation |
| `demand_min` | minimum allowed demand in the truncated distribution |
| `demand_max` | maximum allowed demand in the truncated distribution |
| `num_noise_samples` | default number of demand draws to pre-sample |
| `action_step` | spacing between discrete order quantities |

This is the class to read if you want to know what exact inventory problem the tutorial studies.


### Very Short Summary Of `config.py`

This file does not solve the problem.

It makes the rest of the project easier to read by gathering choices into named bundles.


---
<a id="mdp-file"></a>
## 5. MDP File: `mdp.py`

This file defines the world in which decisions happen.

If `Self_Guided_ALP/falp.py` and `Self_Guided_ALP/sgalp.py` are the “brains,” then `mdp.py` is the “environment.”

Without this file, there is no inventory system to optimize.


### `MarkovDecisionProcess`

Purpose: gives a common interface for any discounted-cost MDP.

Important idea:
- this class is mostly a template,
- it says **what a concrete MDP must know how to do**,
- but it does not itself define a particular inventory model.

| Parameter | Meaning |
| --- | --- |
| `mdp_name` | name of the problem |
| `dim_state` | number of state dimensions |
| `dim_act` | number of action dimensions |
| `discount` | weight on future costs |
| `random_seed` | base random seed |

#### What its methods mean

| Method | Plain-language role |
| --- | --- |
| `get_next_state_given_noise` | compute tomorrow’s state after today’s state, action, and one realized shock |
| `get_cost_given_noise` | compute one realized one-step cost |
| `get_batch_next_state` | do many next-state calculations at once |
| `get_expected_cost` | average cost over many noise draws |
| `sample_noise_batch` | sample exogenous noise |
| `sample_fixed_noise_batch` | save one fixed batch of noise for repeated use |
| `sample_constraint_state_actions` | sample state-action pairs used in ALP constraints |
| `sample_state_relevance_states` | sample states used in the ALP objective |
| `get_batch_init_state` | sample initial states for simulation |
| `get_discrete_actions` | list the available actions |
| `is_state_action_feasible` | check whether a state-action pair is allowed |

There are also a few backward-compatible alias methods. These exist only so older notebook code does not break.


### `UniformSampler`

Purpose: a tiny helper that samples uniformly from an interval.

| Parameter | Meaning |
| --- | --- |
| `low` | lower endpoint |
| `high` | upper endpoint |

#### Method

| Method | Plain-language role |
| --- | --- |
| `rvs` | draw random samples uniformly from the interval |

This class is simple, but it keeps the rest of the code clean.


### Standalone helper functions in `mdp.py`

| Function | Plain-language role |
| --- | --- |
| `positive_part(x)` | returns `x` if `x` is positive, otherwise `0` |
| `draw_truncated_normal(...)` | samples demand from a normal distribution but clips it to stay between a minimum and maximum |
| `_inventory_config_from_dict(...)` | converts an old-style dictionary into the new config object |
| `make_inventory_mdp(...)` | builds the tutorial inventory problem with default settings |

These functions support the main MDP class rather than being the main story themselves.


### `SingleProductInventoryMDP`

Purpose: implements the actual inventory control problem used in the tutorial.

This is the first truly concrete class in the project.

It answers questions like:
- What is the state?
- What happens after I order stock and then demand arrives?
- What costs am I charged?
- What actions are allowed?

| Parameter | Meaning |
| --- | --- |
| `config` | an `InventoryMDPConfig` or equivalent dictionary containing all problem data |

#### Main methods

| Method | Plain-language role |
| --- | --- |
| `clip_inventory` | keeps inventory inside the allowed range |
| `get_next_state_given_noise` | computes next inventory after ordering and demand |
| `get_cost_given_noise` | computes one realized cost |
| `get_batch_next_state` | computes many next states for Monte Carlo averaging |
| `get_expected_cost` | averages the realized costs over sampled demand |
| `sample_noise_batch` | draws demand samples |
| `sample_fixed_noise_batch` | stores one demand batch for reuse |
| `sample_constraint_state_actions` | samples the state-action pairs used in FALP/SGALP constraints |
| `sample_state_relevance_states` | samples states used in the FALP/SGALP objective |
| `get_batch_init_state` | samples starting states for simulation |
| `get_discrete_actions` | returns the order quantities the policy may choose |
| `is_state_action_feasible` | checks basic bounds on state and action |

#### How to think about this class

This class is the “physics” of the problem.

It does **not** choose the policy. It only says what happens if a policy chooses an action.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mdp import make_inventory_mdp

mdp = make_inventory_mdp()
print("State bounds:", mdp.lower_state_bound, mdp.upper_state_bound)
print("First five actions:", mdp.get_discrete_actions()[:5])


---
<a id="basis-file"></a>
## 6. Basis File: `basis.py`

This file defines the building blocks used to approximate the value function.

A simple way to think about it is this:

- the true value function is unknown,
- so we choose a family of flexible curves,
- then we let the algorithm fit the best combination it can within that family.

That family of curves lives here.


### `RandomFourierBasis1D`

Purpose: creates random basis functions for a one-dimensional state.

| Parameter | Meaning |
| --- | --- |
| `max_random_features` | the maximum number of random basis functions to store |
| `config` | a `RandomFeatureConfig` telling the class how to sample them |

#### Main methods

| Method | Plain-language role |
| --- | --- |
| `__init__` | stores the settings and samples the full feature sequence |
| `_sample_feature_sequence` | actually generates the random cosine features |
| `_resolve_num_random_features` | checks how many features should be used right now |
| `eval_basis` | evaluates the basis functions at one state |
| `expected_basis` | averages basis values over many states |
| `get_vfa` | combines basis values with coefficients to produce the approximate value function |

#### Why this file matters

Both FALP and SGALP use this same basis family.

That is important: it means the main difference between FALP and SGALP is **not** the basis. The main difference is how the LP constraints are built and updated.


---
<a id="falp-file"></a>
## 7. FALP File: `Self_Guided_ALP/falp.py`

This file implements the simpler of the two main algorithms.

FALP stands for fitted approximate linear programming.

You do not need to memorize the full theory to read the code. The operational story is enough:

1. choose basis functions,
2. sample many constraints,
3. solve a linear program,
4. get coefficients for an approximate value function.


### `FALP`

Purpose: fit one approximate value function using sampled Bellman-style inequalities.

| Parameter | Meaning |
| --- | --- |
| `mdp` | the inventory problem being solved |
| `config` | preferred way to pass grouped FALP settings |
| `num_random_features` | old-style direct parameter for basis size |
| `num_constraints` | old-style direct parameter for number of sampled constraints |
| `num_state_relevance_samples` | old-style direct parameter for objective sampling |
| `basis_seed` | old-style seed for the basis |
| `bandwidth_choices` | old-style basis-scale choices |
| `solver` | old-style solver selection |

The “old-style” parameters are still accepted for convenience, but the code now prefers using `FALPConfig`.

#### Main methods

| Method | Plain-language role |
| --- | --- |
| `_build_config` | turns old-style scalar arguments into one config object |
| `build_sampled_constraint` | builds one sampled LP inequality from one state-action pair |
| `build_lp` | builds the full sampled LP |
| `solve_lp_by_vertex_enumeration` | tiny teaching solver for very small models |
| `solve_lp_with_scipy` | practical LP solver using SciPy HiGHS |
| `fit` | builds and solves the model |
| `get_falp_objective` | returns the fitted LP’s objective value |
| `estimate_cvl_lower_bound` | asks the lower-bound module to evaluate the fitted model |
| `evaluate_vfa_on_grid` | evaluates the fitted approximation on a state grid |

#### Plain-language lifecycle of a `FALP` object

- First it receives an MDP and some settings.
- Then it creates a basis.
- Then it samples LP constraints.
- Then it solves the LP.
- The solution gives a coefficient vector.
- That coefficient vector defines the approximate value function.


---
<a id="sgalp-file"></a>
## 8. SGALP File: `Self_Guided_ALP/sgalp.py`

This file implements the more advanced algorithm.

The main intuition is simple:

- instead of jumping directly to a large basis,
- SGALP grows the basis in stages,
- and uses guiding constraints so the new stage does not behave too badly compared with the previous stage.

So SGALP is not a totally different universe. It is a staged, guided version of the same basic FALP idea.


### `SelfGuidedALP`

Purpose: fit a sequence of approximate LPs while gradually increasing basis richness.

| Parameter | Meaning |
| --- | --- |
| `mdp` | the inventory problem |
| `config` | preferred grouped SGALP settings |
| `max_random_features` | largest basis size SGALP may reach |
| `batch_size` | number of features added each stage |
| `num_constraints` | sampled Bellman constraints per stage |
| `num_state_relevance_samples` | sampled states used in the objective |
| `num_guiding_states` | sampled states used for guidance |
| `basis_seed` | seed for the shared random-feature sequence |
| `bandwidth_choices` | basis-scale choices |
| `guiding_violation` | direct tolerance for guidance violations |
| `guiding_relax_fraction` | relative relaxation amount |
| `guiding_abs_floor` | minimum relaxation amount |
| `guiding_retry_scales` | backup relaxation levels |
| `highs_method` | LP solver variant |
| `primal_feasibility_tolerance` | solver feasibility tolerance |
| `dual_feasibility_tolerance` | solver dual tolerance |

Again, most of these are grouped more cleanly inside `SGALPConfig`.

#### Main methods

| Method | Plain-language role |
| --- | --- |
| `_build_config` | turns old-style arguments into one config object |
| `build_sampled_constraint` | builds one Bellman-style inequality for a chosen stage size |
| `build_falp_lp` | builds the base LP for a stage |
| `build_guiding_constraints` | adds SGALP’s extra “do not drop too much” constraints |
| `solve_lp` | solves one LP stage |
| `fit_stage` | solves one SGALP stage, with retries if guidance is numerically tight |
| `stage_feature_counts` | returns which feature counts SGALP will solve |
| `fit` | runs the full stage sequence |
| `print_history` | prints a compact summary of stage-by-stage results |
| `evaluate_vfa_on_grid` | evaluates the current fitted approximation on a grid |

#### The single most important conceptual difference from FALP

FALP fits one model.

SGALP fits a chain of models, where each new one is gently guided by the previous one.


---
<a id="lower-bound-file"></a>
## 9. Lower-Bound File: `Self_Guided_ALP/cvl_lower_bound.py`

This file evaluates a fitted model after FALP or SGALP has already done its job.

You can think of it as the file that tries to answer:

**“Can we produce a mathematically meaningful floor on performance?”**

This is not the easiest file in the project, so it is completely normal if this one takes more than one reading.


### `SimpleLNSLowerBound`

Purpose: estimate a sampling-based lower bound for a fitted approximate value function.

| Parameter | Meaning |
| --- | --- |
| `mdp` | the problem being evaluated |
| `basis` | the basis object used by the fitted model |
| `coef` | the fitted coefficient vector |
| `num_random_features` | how many basis functions are active |
| `num_mc_init_states` | number of starting samples for the sampler |
| `chain_length` | number of sampling steps |
| `burn_in` | number of early samples to ignore |
| `proposal_state_std` | proposal step size in the state direction |
| `proposal_action_std` | proposal step size in the action direction |
| `random_seed` | seed for the estimator |
| `noise_batch_size` | number of demand draws reused when estimating residuals |
| `sampler` | whether to use `emcee` or the fallback sampler |
| `num_walkers` | number of `emcee` walkers |
| `initial_state` | starting state for the bound interpretation |

#### Main methods

| Method | Plain-language role |
| --- | --- |
| `get_vfa` | evaluate the fitted approximate value function |
| `get_vfa_batch` | evaluate the approximation for many states at once |
| `saddle_func_batch` | compute the Bellman residual in batch form |
| `saddle_func` | single state-action version of the same idea |
| `get_expected_vfa_on_initial_state` | evaluate the fitted value function at the chosen initial state |
| `get_lipschitz_cost_bound` | compute a problem-specific helper bound |
| `get_lns_constant` | compute a constant used in the bound formula |
| `get_lambda` | compute another tuning constant for the bound |
| `log_target_density` | define the sampler’s target density |
| `log_target_density_batch` | batch version for faster sampling |
| `emcee_sampler` | use `emcee` if available |
| `vectorized_metropolis_sampler` | use the built-in fallback sampler |
| `estimate_lower_bound_stats` | return the lower-bound estimate and diagnostics |
| `estimate_lower_bound` | return only the final lower-bound number |

#### Wrapper functions

| Function | Plain-language role |
| --- | --- |
| `estimate_actual_lower_bound_falp` | lower bound for a fitted FALP model |
| `estimate_actual_lower_bound_sgalp` | lower bound for a fitted SGALP model |

These wrappers exist so the rest of the project can call one easy function instead of manually constructing the estimator every time.


---
<a id="policy-file"></a>
## 10. Policy File: `policy.py`

If the lower-bound file tries to say “performance is at least this good,” the policy file tries to say “this is what the greedy policy actually costs in simulation.”

That is why people often read this file as the upper-bound side of the evaluation story.


### Functions in `policy.py`

| Function | Plain-language role |
| --- | --- |
| `_active_num_random_features(model)` | figures out how many basis functions the model is currently using |
| `build_greedy_policy_lookup(model, config)` | precomputes the greedy action on a dense state grid |
| `estimate_upper_bound_fast(model, config, return_se)` | simulates the greedy policy and averages its total discounted cost |

#### Why the lookup table exists

The policy could recompute the best action from scratch at every simulated state, but that would be slow.

Instead, the code first builds a grid of states and stores the greedy action for each one. Then simulation becomes much faster.


---
<a id="helper-file"></a>
## 11. Helper File: `helper.py`

This is the orchestration file.

It is very useful, but it is not the best first file to learn from.

Why? Because many of its functions assume you already understand the smaller building blocks.

A good way to think about `helper.py` is:

- `mdp.py`, `basis.py`, `Self_Guided_ALP/falp.py`, `Self_Guided_ALP/sgalp.py`, `policy.py`, and `Self_Guided_ALP/cvl_lower_bound.py` define the ingredients,
- `helper.py` is the kitchen workflow that turns them into experiments and plots.


### Main utility functions in `helper.py`

| Function | Plain-language role |
| --- | --- |
| `estimate_falp_objective` | returns the fitted FALP objective |
| `estimate_cvl_lower_bound` | convenience wrapper for lower-bound estimation |
| `build_greedy_policy_lookup` | convenience wrapper for the policy lookup |
| `estimate_upper_bound_falp_fast` | convenience wrapper for upper-bound estimation |
| `moving_average_smoother` | smooths noisy curves |
| `extract_boxplot_stats` | rearranges results into boxplot-friendly form |
| `apply_tutorial_plot_style` | applies one consistent plotting style across notebooks |
| `make_shared_evaluation_configs` | builds shared lower-bound and policy configs |
| `plot_value_function_curves` | draws the reusable FALP and SGALP value-function figure |
| `plot_bound_boxplots` | draws the reusable FALP and SGALP bound boxplots |
| `plot_psmd_iteration_diagnostics` | draws the reusable PSMD iteration comparison figure |
| `plot_psmd_acceptance_and_value` | draws the PSMD acceptance-rate and value-curve figure |
| `plot_psmd_sampling_snapshots` | draws the PSMD state-action snapshot figure |

These are the small notebook-facing helpers that keep the tutorial narrative focused on ideas instead of plotting boilerplate.


### Experiment-running functions in `helper.py`

| Function | Plain-language role |
| --- | --- |
| `run_falp_grid` | runs many FALP models over feature counts and seeds |
| `run_sgalp_grid` | runs many SGALP models over feature counts and seeds |
| `run_psmd_seed_grid` | runs PSMD over multiple seeds with shared reporting logic |
| `run_sgalp_stage_trace` | stores detailed stage-by-stage SGALP information |
| `run_falp_and_sgalp_comparison` | builds a side-by-side comparison object |
| `plot_falp_vs_sgalp_vfas_and_relevance` | creates the final comparison plots |

If you want to know how the notebooks produce their tables and figures without repeating code everywhere, this is the part to read.


### Small config-construction helpers inside `helper.py`

| Function | Plain-language role |
| --- | --- |
| `_make_falp_config` | build or reuse a `FALPConfig` |
| `_make_sgalp_config` | build or reuse a `SGALPConfig` |
| `_make_lower_bound_config` | build or reuse a `LowerBoundConfig` |
| `_make_policy_config` | build or reuse a `PolicyEvaluationConfig` |
| `_make_psmd_config` | build or reuse a `PSMDConfig` |

These functions are not algorithmic stars. They exist to keep the public helper functions neat.


---
<a id="tutorial-notebook"></a>
## 12. Experiment Notebook: `Notebooks/tutorial_self_guided_ALP.ipynb`

This notebook is the public-facing demonstration notebook.

Its job is to:
- import the code,
- choose experiment settings,
- run FALP and SGALP,
- plot the results,
- explain what the plots mean.

In other words, the `.py` files are the engine room, and `Notebooks/tutorial_self_guided_ALP.ipynb` is the presentation deck.


---
## How the files link to each other

A very simple dependency picture is:

- `config.py` feeds settings into almost everything.
- `mdp.py` provides the inventory problem.
- `basis.py` provides the approximation family.
- `Self_Guided_ALP/falp.py` and `Self_Guided_ALP/sgalp.py` combine the MDP and basis to fit value functions.
- `Self_Guided_ALP/cvl_lower_bound.py` evaluates fitted models from `Self_Guided_ALP/falp.py` or `Self_Guided_ALP/sgalp.py`.
- `policy.py` also evaluates fitted models from `Self_Guided_ALP/falp.py` or `Self_Guided_ALP/sgalp.py`.
- `helper.py` calls all of the above to run experiments.
- `Notebooks/tutorial_self_guided_ALP.ipynb` calls `helper.py` to present results.


---
<a id="end-to-end-story"></a>
## 13. Short End-to-End Story

Here is the whole project in plain language.

1. We define an inventory system in `mdp.py`.
2. We choose a flexible family of approximation curves in `basis.py`.
3. We fit those curves with FALP or SGALP in `Self_Guided_ALP/falp.py` or `Self_Guided_ALP/sgalp.py`.
4. We ask, “How good is the fitted approximation?”
5. We estimate a lower bound in `Self_Guided_ALP/cvl_lower_bound.py`.
6. We estimate an upper bound by simulation in `policy.py`.
7. We automate repeated runs and plots in `helper.py`.
8. We present the whole story in `Notebooks/tutorial_self_guided_ALP.ipynb`.

That is the codebase.

If you remember nothing else, remember this:

- `mdp.py` is the problem,
- `basis.py` is the approximation family,
- `Self_Guided_ALP/falp.py` and `Self_Guided_ALP/sgalp.py` are the fitting algorithms,
- `Self_Guided_ALP/cvl_lower_bound.py` and `policy.py` are the evaluators,
- `helper.py` is the experiment manager.


---
<a id="end-to-end-story"></a>
## 14. If You Want To Study The Code Carefully

A practical study plan is:

1. Read the config classes and write down what each knob changes.
2. Read `SingleProductInventoryMDP` and make sure you understand the state, action, demand, and cost.
3. Read `RandomFourierBasis1D` and understand that it turns a state into a vector of numbers.
4. Read `FALP.fit()` and trace what happens line by line.
5. Read `SelfGuidedALP.fit()` and compare it with `FALP.fit()`.
6. Only after that, study the lower-bound and policy-evaluation files.

That order usually reduces confusion a lot.
